# Acquisition earnout analysis

This notebook values the contingent earnout in the small TargetCo acquisition case.
The earnout is intentionally stylized: a cash-settled capped payout on a normalized
KPI index. That lets the analysis reuse the platform's existing Black-Scholes
closed-form and Monte Carlo European-option infrastructure as a call spread rather
than adding a new model.

The Excel workbook owns the transparent operating forecast and DCF. This notebook
owns the quantitative earnout evidence, convergence checks, sensitivity analysis,
and validation boundaries.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, replace
from datetime import date
from math import sqrt

from qf_platform.pricing import (
    BlackScholesClosedForm,
    BlackScholesLaw,
    BlackScholesParameters,
    EquityState,
    EuropeanOption,
    FlatMoneyMarketNumeraire,
    ModeledState,
    MonteCarloEuropeanOption,
    OptionRight,
    PricingMeasureSemantics,
    PricingProblem,
    actual_365_fixed_year_fraction,
)


In [ ]:
@dataclass(frozen=True, slots=True)
class EarnoutTerms:
    """Stylized earnout represented as a capped call spread on a KPI index."""

    valuation_date: date
    payout_date: date
    kpi_index_now: float
    start_strike: float
    cap_strike: float
    max_payout: float
    risk_free_rate: float
    kpi_volatility: float
    seed: int = 1729

    @property
    def scale(self) -> float:
        return self.max_payout / (self.cap_strike - self.start_strike)


terms = EarnoutTerms(
    valuation_date=date(2026, 9, 22),
    payout_date=date(2028, 9, 21),
    kpi_index_now=100.0,
    start_strike=115.0,
    cap_strike=135.0,
    max_payout=80.0,
    risk_free_rate=0.0425,
    kpi_volatility=0.25,
)

actual_365_fixed_year_fraction(terms.valuation_date, terms.payout_date)


In [ ]:
def make_call_problem(terms: EarnoutTerms, strike: float) -> PricingProblem[date, EquityState, BlackScholesParameters]:
    """Map the KPI-index call into the existing Black-Scholes pricing problem."""
    law = BlackScholesLaw()
    numeraire = FlatMoneyMarketNumeraire(
        terms.valuation_date,
        terms.risk_free_rate,
    )
    return PricingProblem(
        current_state=ModeledState(
            terms.valuation_date,
            EquityState(terms.kpi_index_now),
            law.state_space,
        ),
        stochastic_law=law,
        parameters=BlackScholesParameters(
            annualized_volatility=terms.kpi_volatility,
        ),
        contract=EuropeanOption(terms.payout_date, strike, OptionRight.CALL),
        numeraire=numeraire,
        pricing_measure=PricingMeasureSemantics(name="Q^B", numeraire=numeraire),
    )


def platform_call_closed_form(terms: EarnoutTerms, strike: float) -> float:
    return BlackScholesClosedForm().apply(make_call_problem(terms, strike)).present_value


def platform_call_monte_carlo(
    terms: EarnoutTerms,
    strike: float,
    *,
    paths: int,
) -> tuple[float, float]:
    result = MonteCarloEuropeanOption(paths=paths, seed=terms.seed).apply(
        make_call_problem(terms, strike),
    )
    return result.present_value, result.standard_error


def earnout_closed_form_value(terms: EarnoutTerms) -> float:
    lower_call = platform_call_closed_form(terms, terms.start_strike)
    upper_call = platform_call_closed_form(terms, terms.cap_strike)
    return terms.scale * (lower_call - upper_call)


def earnout_monte_carlo_value(
    terms: EarnoutTerms,
    *,
    paths: int,
) -> tuple[float, float]:
    lower_call, lower_se = platform_call_monte_carlo(
        terms,
        terms.start_strike,
        paths=paths,
    )
    upper_call, upper_se = platform_call_monte_carlo(
        terms,
        terms.cap_strike,
        paths=paths,
    )
    value = terms.scale * (lower_call - upper_call)
    conservative_se = terms.scale * sqrt(lower_se * lower_se + upper_se * upper_se)
    return value, conservative_se

closed_form_value = earnout_closed_form_value(terms)
closed_form_value


## Convergence

The closed-form call spread is the reference value because the earnout has been
mapped into a European option structure. Monte Carlo is used here as an independent
sampling check through the platform's existing seeded Monte Carlo method.


In [ ]:
convergence_rows: list[dict[str, float | int]] = []
for paths in (2_000, 10_000, 40_000):
    estimate, standard_error = earnout_monte_carlo_value(terms, paths=paths)
    convergence_rows.append(
        {
            "paths": paths,
            "estimate": estimate,
            "standard_error": standard_error,
            "abs_error_vs_closed_form": abs(estimate - closed_form_value),
            "ci_half_width_95": 1.959963984540054 * standard_error,
        },
    )

convergence_rows


## Sensitivity

The earnout value should increase with KPI volatility for this out-of-the-money
capped upside payout, and it should decrease as the start hurdle rises.


In [ ]:
volatility_sensitivity = [
    {
        "kpi_volatility": volatility,
        "earnout_value": earnout_closed_form_value(
            replace(terms, kpi_volatility=volatility),
        ),
    }
    for volatility in (0.15, 0.20, 0.25, 0.30, 0.35)
]

hurdle_sensitivity = [
    {
        "start_strike": start_strike,
        "cap_strike": start_strike + 20.0,
        "earnout_value": earnout_closed_form_value(
            replace(
                terms,
                start_strike=start_strike,
                cap_strike=start_strike + 20.0,
            ),
        ),
    }
    for start_strike in (105.0, 115.0, 125.0)
]

volatility_sensitivity, hurdle_sensitivity


## Validation checks and boundaries

These checks guard numerical and economic interpretation only. They do not prove
that the KPI follows Black-Scholes dynamics or that the earnout is fairly negotiated.
The case is a portfolio artifact showing modeling flow, traceability, and model-risk
communication.


In [ ]:
assert 0.0 <= closed_form_value <= terms.max_payout
assert platform_call_closed_form(terms, terms.start_strike) >= platform_call_closed_form(
    terms,
    terms.cap_strike,
)

last_convergence = convergence_rows[-1]
assert abs(float(last_convergence["estimate"]) - closed_form_value) < max(
    3.0 * float(last_convergence["ci_half_width_95"]),
    2.0,
)

vol_values = [float(row["earnout_value"]) for row in volatility_sensitivity]
assert vol_values == sorted(vol_values)

hurdle_values = [float(row["earnout_value"]) for row in hurdle_sensitivity]
assert hurdle_values == sorted(hurdle_values, reverse=True)

summary = {
    "closed_form_earnout_value_mm": closed_form_value,
    "final_mc_estimate_mm": last_convergence["estimate"],
    "max_payout_mm": terms.max_payout,
    "valuation_boundary": "stylized KPI-index call spread; not empirical KPI model validation",
}
summary
